---

<h3><center>E178/ME276DS - Statistics and Data Science for Engineers</center></h3>

<h1><center>Final Project Data Cleaning<br></center></h1>

---

In [10]:
import numpy as np
import pandas as pd
import scipy.stats as stats
import matplotlib.pyplot as plt



In [17]:
data_PressureLevel = pd.read_csv('Data/Pressure Level Data.csv')
data_CloudandRain = pd.read_csv('Data/Cloud Coverage and Precipitation.csv')

In [18]:
print(data_PressureLevel.columns)
print(data_PressureLevel.head(5))
print('-'*90)
print(data_CloudandRain.columns)
print(data_CloudandRain.head(5))
print('-'*90)



Index(['valid_time', 'pressureLevel', 'z', 'u', 'v', 'r', 't', 'latitude',
       'longitude'],
      dtype='object')
     valid_time  pressureLevel           z         u         v          r  \
0  3/1/95 00:00           1000   1382.3281  1.648590 -0.128830  55.401173   
1  3/1/95 00:00            925   7822.1400  1.654526 -0.131912  55.169790   
2  3/1/95 00:00            850  14707.0625  1.616882 -0.123596  55.593460   
3  3/1/95 00:00            700  30132.7770  1.595703 -0.106598  54.917370   
4  3/1/95 00:00            600  42007.5860  1.717270  0.065689  52.694664   

           t  latitude  longitude  
0  292.19580        28         87  
1  287.90283        28         87  
2  283.36716        28         87  
3  273.04413        28         87  
4  265.12560        28         87  
------------------------------------------------------------------------------------------
Index(['valid_time', 'tcc', 'tp', 'latitude', 'longitude'], dtype='object')
     valid_time       tcc        tp 

In [19]:
# Care about 300 and 400 Pressure Level
PL_mask = data_PressureLevel['pressureLevel'].between(290,410)
data_PressureLevel = data_PressureLevel.loc[PL_mask,:].reset_index(drop = True)
data_PressureLevel['r'] = data_PressureLevel['r']/ 100

In [20]:
data_PressureLevel['valid_time'] = pd.to_datetime(data_PressureLevel['valid_time'], errors = 'coerce',format ='mixed')
data_CloudandRain['valid_time'] = pd.to_datetime(data_CloudandRain['valid_time'], errors = 'coerce',format='mixed')

In [21]:
# Limtied to every six hours by Pressure Level Data, match hourly data to every 6 hours
time_mask = data_CloudandRain['valid_time'].dt.hour.isin([0,6,12,18])
data_CloudandRain = data_CloudandRain.loc[time_mask,:].reset_index(drop = True)

In [22]:
# print(data_PressureLevel.shape)
# print(data_PressureLevel.info)
# print('-'*50)
# print(data_CloudandRain.shape)
# print(data_CloudandRain.info)

In [29]:

# u and v wind speed to max wind speed
data_PressureLevel['max_wind_speed'] =np.sqrt( data_PressureLevel['u'] ** 2 + data_PressureLevel['v'] **2 ) 
data_PressureLevel.drop(columns= ['u','v'])


,valid_time,pressureLevel,z,r,t,latitude,longitude,max_wind_speed
0,1995-03-01 00:00:00,400,71847.530,0.041893,249.90230,28,87,30.434310
1,1995-03-01 00:00:00,300,92209.250,0.049329,242.74450,28,87,65.047543
2,1995-03-01 06:00:00,400,72200.710,0.038382,250.55923,28,87,29.867733
3,1995-03-01 06:00:00,300,92606.125,0.055107,242.77657,28,87,63.409191
4,1995-03-01 12:00:00,400,72245.750,0.067576,249.99225,28,87,28.944025
...,...,...,...,...,...,...,...,...
91555,2026-06-30 06:00:00,300,95974.875,0.461204,249.65164,28,87,1.849989
91556,2026-06-30 12:00:00,400,74713.800,0.598950,263.71857,28,87,6.784114
91557,2026-06-30 12:00:00,300,95971.500,0.330212,250.05530,28,87,7.297415
91558,2026-06-30 18:00:00,400,74759.030,0.761330,262.34192,28,87,2.687073


In [27]:
# 1. Pivot pressure levels (300 and 400) so each timestamp is 1 row
vars_to_pivot = ['z', 'r', 't','max_wind_speed']

df_pressure_wide = data_PressureLevel.pivot(
    index=['valid_time', 'latitude', 'longitude'],
    columns='pressureLevel',
    values=vars_to_pivot
)

# 2. Flatten MultiIndex columns into clean names (e.g., 'z_300', 'z_400', 't_300', 't_400')
df_pressure_wide.columns = [f"{var}_{int(lvl)}" for var, lvl in df_pressure_wide.columns]
df_pressure_wide = df_pressure_wide.reset_index()

# 3. Splice the two DataFrames together on valid_time, latitude, and longitude
merged_data = pd.merge(
    data_CloudandRain,
    df_pressure_wide,
    on=['valid_time', 'latitude', 'longitude'],
    how='inner'
)

merged_data = merged_data.drop(columns=['latitude','longitude'])
print(merged_data.shape)
merged_data.head(10)

(45780, 11)


,valid_time,tcc,tp,z_300,z_400,r_300,r_400,t_300,t_400,max_wind_speed_300,max_wind_speed_400
0,1995-03-01 00:00:00,0.064667,0.000034,92209.250,71847.530,0.049329,0.041893,242.74450,249.90230,65.047543,30.434310
1,1995-03-01 06:00:00,0.079956,0.000025,92606.125,72200.710,0.055107,0.038382,242.77657,250.55923,63.409191,29.867733
2,1995-03-01 12:00:00,0.053314,0.000003,92566.190,72245.750,0.063069,0.067576,242.20369,249.99225,61.289830,28.944025
3,1995-03-01 18:00:00,0.009338,0.000000,92966.440,72589.484,0.060122,0.157725,243.27147,249.72462,60.530956,28.784990
4,1995-03-02 00:00:00,0.041992,0.000003,92880.875,72472.310,0.049564,0.054815,242.28711,250.93219,56.528526,35.548845
5,1995-03-02 06:00:00,0.063965,0.000021,93201.625,72732.170,0.052975,0.043462,242.41414,252.16872,55.977149,40.097436
6,1995-03-02 12:00:00,0.258728,0.000021,92948.940,72527.580,0.044513,0.058761,242.71840,251.52496,59.117288,41.141356
7,1995-03-02 18:00:00,0.177307,0.000029,92944.690,72547.695,0.041528,0.045742,243.35085,250.48760,62.676215,39.945596
8,1995-03-03 00:00:00,0.092651,0.000015,92666.310,72404.240,0.040959,0.093553,242.04222,249.82149,61.667803,38.453006
9,1995-03-03 06:00:00,0.102631,0.000030,92741.440,72633.740,0.109552,0.112957,239.19885,250.12581,53.866116,38.005397


In [28]:
try:
    merged_data.to_csv('Data/Merged Weather.csv')
except: 
    raise('BufferError')